In [ ]:
import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")

In [ ]:
from typing import Annotated, Literal, Sequence, TypedDict
from langchain import hub
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langgraph.graph.message import add_messages
from langgraph.prebuilt import tools_condition
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.tools.retriever import create_retriever_tool
from langgraph.graph import END, StateGraph, START
from langgraph.prebuilt import ToolNode

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY=os.getenv("LANGCHAIN_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY
os.environ["GROQ_API_KEY"]= GROQ_API_KEY
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

##### it is having details about the tool condition we build our custom logic also

https://github.com/langchain-ai/langgraph/blob/main/libs/langgraph/langgraph/prebuilt/tool_node.py

why we use from langchain_core.pydantic_v1 import BaseModel, Field

The import statement:

```python
from langchain_core.pydantic_v1 import BaseModel, Field
```

indicates that you're working within the **LangChain ecosystem** and leveraging **Pydantic** for data validation and model creation. Here's a breakdown of **why and how** these components are used:

---

### **Why Use `BaseModel` and `Field` from Pydantic?**

Pydantic is a powerful library for data validation and settings management in Python. It is particularly well-suited for building models that ensure your data conforms to specific types and formats.

In the context of LangChain, **Pydantic's `BaseModel` and `Field`** are often used to define structured schemas for:

1. **Tool Input and Output Models:**  
   - Define the input and output structure for tools or LLM-based functions.
   - Ensures the data being passed to or returned from the tools is valid and well-defined.

2. **Prompts and Contexts:**  
   - Create reusable, structured representations of prompts or conversation states.

3. **Validation and Type Enforcement:**  
   - Validate that data passed between components (e.g., tools, LLMs, and workflows) adheres to expected types and constraints.

4. **Better Readability and Debugging:**  
   - Explicitly define schemas for tools and workflows, making your code easier to understand and debug.

---

### **How These Components Work**

#### **1. `BaseModel`**
Pydantic's `BaseModel` is the foundation for creating structured models. It allows you to:
- Define the expected fields of a model.
- Add validation logic for those fields.
- Automatically handle type coercion and validation errors.

#### **2. `Field`**
The `Field` class is used to:
- Add metadata, constraints, or default values to model fields.
- Document the purpose of each field.

---

### **Examples**

#### **Tool Input Model**
Suppose you are defining a tool that takes user input. You can use `BaseModel` and `Field` to structure the expected input:

```python
from langchain_core.pydantic_v1 import BaseModel, Field

class MathToolInput(BaseModel):
    number1: float = Field(..., description="The first number for the operation")
    number2: float = Field(..., description="The second number for the operation")
    operation: str = Field(..., description="The operation to perform: add, subtract, multiply, or divide")

# Example usage
input_data = MathToolInput(number1=10, number2=5, operation="add")
print(input_data.dict())  # {'number1': 10.0, 'number2': 5.0, 'operation': 'add'}
```

This ensures:
- Only valid operations (e.g., "add") are processed.
- Invalid or missing data raises a clear validation error.

---

#### **Structured Prompts**
You can use `BaseModel` to create reusable, structured prompts for LLMs:

```python
class SummarizationPrompt(BaseModel):
    text: str = Field(..., description="The text to summarize")
    max_length: int = Field(100, description="Maximum length of the summary")

prompt = SummarizationPrompt(text="This is a long article...", max_length=50)
print(prompt.dict())  # {'text': 'This is a long article...', 'max_length': 50}
```

---

#### **Tool Input Validation**
You can integrate these models directly into LangChain tools:

```python
from langchain_core.pydantic_v1 import BaseModel
from langchain.tools import BaseTool

class RetrieveBlogPostsInput(BaseModel):
    query: str = Field(..., description="The search query for blog posts")
    max_results: int = Field(10, description="The maximum number of blog posts to return")

class RetrieveBlogPostsTool(BaseTool):
    name = "retrieve_blog_posts"
    description = "Retrieve blog posts based on a search query"
    args_schema = RetrieveBlogPostsInput  # Link the schema

    def _run(self, query: str, max_results: int):
        return f"Retrieved {max_results} results for query: {query}"

# Example usage
tool = RetrieveBlogPostsTool()
print(tool.run(query="AI advancements", max_results=5))  # Retrieves the results
```

---

### **Benefits of Using `BaseModel` and `Field`**

1. **Type Safety:**  
   - Ensures data integrity across components.
   
2. **Descriptive Errors:**  
   - Automatically raises clear validation errors for invalid input.

3. **Reusability:**  
   - Define input/output models once and reuse them in multiple tools or workflows.

4. **Self-Documentation:**  
   - The `Field` metadata (e.g., `description`) makes your code easier to understand and maintain.

5. **Interoperability:**  
   - These structured models are compatible with LangChain's tooling and workflows, making them a natural choice in this ecosystem.

---

Let me know if you’d like help with implementing these models for your specific use case! 😊

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

In [ ]:
import os

# Load API key from environment variable or hardcode (not recommended for production)
# Example: Groq API key
os.environ["GROQ_API_KEY"] = ""

# If HuggingFace requires a token for private models:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""

# Now you can safely import and initialize the models
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

from langchain_groq import ChatGroq
llm = ChatGroq(model="Gemma2-9b-It")


In [ ]:
llm.invoke("hi hello how are you?")

In [ ]:
url= "https://lilianweng.github.io/posts/2023-06-23-agent/"

In [ ]:
pip install beautifulsoup4


In [ ]:
from langchain.document_loaders import WebBaseLoader

url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
docs = WebBaseLoader(url).load()


In [ ]:
WebBaseLoader(url).load()[0].metadata

In [ ]:
description=WebBaseLoader(url).load()[0].metadata["description"]

In [ ]:
description

In [ ]:
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
]

In [ ]:
docs = [WebBaseLoader(url).load() for url in urls]

In [ ]:
docs

In [ ]:
docs_list = [item for sublist in docs for item in sublist]

In [ ]:
docs_list

In [ ]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=100, chunk_overlap=5)

In [ ]:
doc_splits=text_splitter.split_documents(docs_list)

In [ ]:
doc_splits

In [ ]:
pip install chromadb

In [ ]:
vectorstore=Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag-chrome",
    embedding=embeddings
    
)

In [ ]:
retriever=vectorstore.as_retriever()

In [ ]:
retriever_tool = create_retriever_tool(
    retriever,
    "retrieve_blog_posts",
    "Search and return information about Lilian Weng blog posts on LLM agents, prompt engineering, and adversarial attacks on LLMs.You are a specialized assistant. Use the 'retriever_tool' **only** when the query explicitly relates to LangChain blog data. For all other queries, respond directly without using any tool. For simple queries like 'hi', 'hello', or 'how are you', provide a normal response.",
    
)

In [ ]:
tools=[retriever_tool]

In [ ]:
retrieve=ToolNode([retriever_tool])

In [ ]:
llm_with_tools=llm.bind_tools(tools)

In [ ]:
llm_with_tools.invoke(["hi how are you can you tell me what is india's capital?"])

In [ ]:
llm_with_tools.invoke(["who is a prime minister of india?"])

In [ ]:
llm_with_tools.invoke(["hi can you tell me what is llm agent?"])

In [ ]:
llm_with_tools.invoke(["How does short-term memory (in-context learning) differ from long-term memory in an LLM-powered autonomous agent system?"])

Certainly! Let's break down the code:

### **Code Breakdown:**

```python
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
```

This code defines a class `AgentState` that is a **`TypedDict`**, which is a special kind of dictionary with fixed keys and value types, used to provide better type safety and autocompletion in Python.

Here’s a breakdown of each part:

---

### **1. `AgentState(TypedDict)`**
- **`TypedDict`**: This is a special class provided by `typing` that allows you to define dictionaries where each key has a fixed type. The dictionary keys are usually predefined, and each key must have a specified type.

  - `AgentState` here is a subclass of `TypedDict`, meaning `AgentState` will represent a dictionary where the types of its values are enforced.

  - Example:
    ```python
    agent_state = AgentState(messages=["Message 1", "Message 2"])  # This would be valid.
    ```

---

### **2. `messages: Annotated[Sequence[BaseMessage], add_messages]`**
- **`messages`**: This is a key in the `AgentState` dictionary. It holds a sequence (like a list or tuple) of `BaseMessage` objects.
  
- **`Annotated[...]`**: `Annotated` is used to attach additional metadata or annotations to the type. It’s a way to add extra information to the type system without affecting the actual type. In this case, it provides metadata for the `messages` field. 

  - The **`Annotated`** wrapper allows you to define a type and also include additional information that might be used by tools like static analyzers or for other purposes like validation.

- **`Sequence[BaseMessage]`**: This indicates that `messages` should be a sequence (e.g., a list or tuple) of `BaseMessage` objects.

  - **`BaseMessage`** is likely a class that represents a message in the context of your program. It could have fields like `content`, `sender`, `timestamp`, etc.
  
- **`add_messages`**: This is a special annotation (or metadata) that likely refers to some function or constant that’s being added as additional context. The specific role of `add_messages` will depend on how it's defined elsewhere in the codebase.

  - **`add_messages`** could be a function, validator, or modifier that helps manage or process the messages in some way, but it's not directly modifying the type of `messages`. It's simply metadata for the type checker or other parts of the system.

  - For example, if `add_messages` were a function, it could be used in a custom decorator, validation system, or simply as an indication of how to handle messages.

---

### **Putting It All Together:**

- **`AgentState`** is a dictionary where one of the keys, `messages`, is specifically typed to be a **sequence of `BaseMessage`** objects. The sequence is annotated with **`add_messages`**, which provides additional metadata but doesn’t affect the type directly.
  
- **Example of `AgentState` usage:**

```python
from typing import TypedDict, Sequence
from typing_extensions import Annotated

class BaseMessage:
    def __init__(self, content: str):
        self.content = content

# Defining the `AgentState` TypedDict
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], "add_messages"]

# Example usage
message1 = BaseMessage("Hello, world!")
message2 = BaseMessage("How are you?")
state = AgentState(messages=[message1, message2])

print(state)
```

In the above code:
- `AgentState` is a dictionary with a `messages` field that is a sequence of `BaseMessage` objects.
- `add_messages` is metadata associated with the `messages` field, which can help guide the usage or interpretation of the data.

---

### **Summary:**
- `AgentState` is a dictionary where the `messages` key holds a sequence of `BaseMessage` objects.
- `Annotated[Sequence[BaseMessage], add_messages]` means that `messages` is a sequence of `BaseMessage` objects with additional metadata (`add_messages`).
- The annotation `add_messages` provides extra context or metadata that doesn't alter the type directly but can be used for purposes like validation or other system-specific behavior.

Let me know if you'd like further clarification on any part of this! 😊

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [ ]:
'''def handle_query(query):
    print(query[0])
    if "prompt" in query[0] or "agent" in query[0]:
        return llm_with_tools.invoke(query)  # Use tool-bound LLM
    else:
        return llm.invoke(query)  # Use LLM directly without tool'''

In [ ]:
def ai_assistant(state:AgentState):
    print("---CALL AGENT---")
    messages = state['messages']
    print(f"this is my message: {messages}")
    
    if len(messages)>1:
        response=llm.invoke(messages[-1].content)
        return {"messages": [response]}
    else:
        llm_with_tool = llm.bind_tools(tools)
        response = llm_with_tool.invoke(messages)
        #response=handle_query(messages)
        return {"messages": [response]}

In [ ]:
{"messages":["hi hello how are you?"]}

In [ ]:
ai_assistant({"messages":["hi hello how are you?"]})

In [ ]:
ai_assistant({"messages":["what is autonomous agent"]})

Let's break down the code:

```python
class grade(BaseModel):
    binary_score: str = Field(description="Relevance score 'yes' or 'no'")
```

This code defines a **Pydantic model** called `grade`, which is a subclass of **`BaseModel`**. The model is used to enforce the structure and data validation for a specific object. Here’s a detailed breakdown of each part:

---

### **1. `class grade(BaseModel):`**
- **`BaseModel`**: This is a class from the **Pydantic** library. By subclassing `BaseModel`, the `grade` class inherits Pydantic's functionality, which includes data validation, parsing, and automatic type checks.
  - When you define a model with Pydantic's `BaseModel`, it ensures that the data being processed conforms to the defined types and validates the fields according to their specified types.

- **`grade`**: This is the name of the model. It represents a grading system (or some other form of assessment) with a specific field, `binary_score`.

---

### **2. `binary_score: str = Field(description="Relevance score 'yes' or 'no'")`**
- **`binary_score`**: This is a field in the `grade` model. It holds the score related to relevance.
  - It is typed as a `str` (string) and is expected to hold a value of `'yes'` or `'no'`.

- **`Field(description="Relevance score 'yes' or 'no'")`**:  
  - **`Field`**: This is a Pydantic function that is used to add metadata to model fields. In this case, it describes the field in the model.
  - The **`description`** argument provides a brief explanation of what the field represents, which helps improve documentation and readability of the code. This metadata doesn't change the field's behavior but can be helpful for generating documentation or providing context.

---

### **What Does This Model Do?**
- **The `grade` model** represents an object where the `binary_score` field holds a string value, specifically `'yes'` or `'no'`, indicating a relevance score.
- The **`description`** metadata is useful for documenting that this field specifically holds a binary score, either 'yes' or 'no'.

---

### **Example Usage**

You can create an instance of this model and validate the data automatically, thanks to Pydantic:

```python
from pydantic import BaseModel, Field

# Define the grade model
class grade(BaseModel):
    binary_score: str = Field(description="Relevance score 'yes' or 'no'")

# Example usage: create an instance of grade
score = grade(binary_score="yes")
print(score)
```

#### Output:
```python
grade(binary_score='yes')
```

- If you try to create an instance with a value other than `'yes'` or `'no'`, Pydantic will raise a validation error, because the type and values are strictly defined.

#### Invalid Example:
```python
score = grade(binary_score="maybe")  # Raises validation error
```

#### Error:
```python
pydantic.error_wrappers.ValidationError: 1 validation error for grade
binary_score
  value is not a valid string (type=value_error.str)
```

---

### **Summary:**
- The **`grade` model** defines a **binary relevance score** (`binary_score`), where the score should be `'yes'` or `'no'`.
- **Pydantic's `Field`** is used to add a description to this field, explaining its purpose.
- By subclassing `BaseModel`, this model automatically provides data validation, ensuring that the `binary_score` is a valid string and provides automatic error handling if invalid data is provided.

Let me know if you'd like further explanation or examples! 😊

In [ ]:
class grade(BaseModel):
    binary_score:str=Field(description="Relevance score 'yes' or 'no'")

In [ ]:
def grade_documents(state:AgentState)->Literal["Output_Generator", "Query_Rewriter"]:
    llm_with_structure_op=llm.with_structured_output(grade)
    
    prompt=PromptTemplate(
        template="""You are a grader deciding if a document is relevant to a user’s question.
                    Here is the document: {context}
                    Here is the user’s question: {question}
                    If the document talks about or contains information related to the user’s question, mark it as relevant. 
                    Give a 'yes' or 'no' answer to show if the document is relevant to the question.""",
                    input_variables=["context", "question"]
                    )
    chain = prompt | llm_with_structure_op
    
    messages = state["messages"]
    print(f"message from the grader: {messages}")
    last_message = messages[-1]
    question = messages[0].content
    docs = last_message.content
    scored_result = chain.invoke({"question": question, "context": docs})
    score = scored_result.binary_score

    if score == "yes":
        print("---DECISION: DOCS RELEVANT---")
        return "generator" #this should be a node name
    else:
        print("---DECISION: DOCS NOT RELEVANT---")
        return "rewriter" #this should be a node name

### from langchain hub we can fetch the prompt

In [ ]:
prompt = hub.pull("rlm/rag-prompt")

In [ ]:
prompt.pretty_print()

In [ ]:
def generate(state:AgentState):
    print("---GENERATE---")
    messages = state["messages"]
    
    print(f"here is message from generate: {messages}")
    
    question = messages[0].content
    last_message = messages[-1]
    docs = last_message.content
    
    prompt = hub.pull("rlm/rag-prompt")
    
    rag_chain = prompt | llm

    response = rag_chain.invoke({"context": docs, "question": question})
    print(f"this is my response:{response}")
    
    return {"messages": [response]}

In [ ]:
from langchain_core.messages import  HumanMessage
def rewrite(state:AgentState):
    print("---TRANSFORM QUERY---")
    messages = state["messages"]
    question = messages[0].content
    
    print(f"here is message from rewrite: {messages}")
    
    message = [HumanMessage(content=f"""Look at the input and try to reason about the underlying semantic intent or meaning. 
                    Here is the initial question: {question} 
                    Formulate an improved question: """)
       ]
    response = llm.invoke(message)
    return {"messages": [response]}

In [ ]:
workflow=StateGraph(AgentState)
workflow.add_node("My_AI_Assistant",ai_assistant)
workflow.add_node("Vector_Retriever", retrieve) 
workflow.add_node("Query_Rewriter", rewrite) 
workflow.add_node("Output_Generator", generate)

In [ ]:
workflow.add_edge(START,"My_AI_Assistant")

In [ ]:
workflow.add_conditional_edges("My_AI_Assistant",
                               tools_condition,
                               {"tools": "Vector_Retriever",
                                END: END,})

In [ ]:
workflow.add_conditional_edges("Vector_Retriever",
                            grade_documents,
                            {"generator": "Output_Generator",
                            "rewriter": "Query_Rewriter"
                            }
                            )
                            

In [ ]:
workflow.add_edge("Output_Generator", END)

In [ ]:
workflow.add_edge("Query_Rewriter", "My_AI_Assistant")

In [ ]:
app=workflow.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
app.invoke({"messages":["What is a Autonomous Agent?"]})

In [ ]:
app.invoke({"messages":["What is a prompt engineering?"]})

In [ ]:
app.invoke({"messages":["who is a current prime minister of india?"]})

In [ ]:
app.invoke({"messages":["what is solar eclips?"]})

In [ ]:
app.invoke({"messages":["how we can use data structure for fundamental part of planning?"]})

In [ ]:
app.invoke({"messages":["what finance minister does for fundamental planning?"]})

This is a fundamental part of planning, breaking down complex tasks into smaller, manageable ones, often used in agent systems for better execution.